# Rainfall Regime Analysis for Agricultural Extension
Compares monthly rainfall across **Kampala, Gulu and Mbarara** and classifies month-by-month suitability for **maize, beans and coffee (Robusta)**.

**Data note:** the monthly series are the *illustrative* figures from the task brief, and the 10-year section is *synthetic*. Neither is measured data; replace with NASA POWER / UNMA records before using any result for planting decisions.

All reusable logic lives in `src/`; this notebook only calls it and shows results.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # run the notebook from the project root

import pandas as pd
import matplotlib.pyplot as plt

from src.data import MONTHS, REGIONS, region_summary
from src.crops import CROP_RULES, classify_all
from src.similarity import verify_cosine_against_scipy, build_matrices
from src.seasonality import classify_modality
from src.synthetic import generate_all
from src import plots

FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
%matplotlib inline

## 1. Region statistics

In [ ]:
pd.DataFrame(region_summary()).set_index('Region')

## 2. Monthly rainfall by region

In [ ]:
fig = plots.line_chart()
fig.savefig(FIG_DIR / "rainfall_line_chart.png", dpi=150)
plt.show()

## 3. Crop suitability
Thresholds and their sources are documented in `src/crops.py`.

In [ ]:
pd.DataFrame(
    [{"Crop": c, "Good min": r.good_min, "Good max": r.good_max,
      "Waterlog >": r.waterlog_max, "Source": r.source}
     for c, r in CROP_RULES.items()]
).set_index("Crop")

In [ ]:
classification = classify_all()
pd.concat({r: pd.DataFrame(crops, index=MONTHS).T for r, crops in classification.items()})

In [ ]:
fig = plots.suitability_heatmap()
fig.savefig(FIG_DIR / "crop_suitability_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Similarity between regions
Cosine similarity is the proper vector definition (dot product over the product of norms), fixing last year's `math.cos()` bug. It is verified against SciPy's cosine *distance* (similarity = 1 − distance).

In [ ]:
pd.DataFrame(verify_cosine_against_scipy())

In [ ]:
names, cos_mat, pearson_mat, eucl_mat = build_matrices()
def as_df(m): return pd.DataFrame(m, index=names, columns=names).round(2)

print("Cosine similarity (shape/timing, ignores magnitude)");         display(as_df(cos_mat))
print("Pearson correlation (seasonal pattern, mean-centred)");        display(as_df(pearson_mat))
print("Euclidean distance in mm (0 = identical)");                    display(as_df(eucl_mat))

## 5. Seasonality detection

In [ ]:
pd.DataFrame(
    [{"Region": n, "Modality": classify_modality(r)[0],
      "Peak months": ", ".join(classify_modality(r)[1])}
     for n, r in REGIONS.items()]
).set_index("Region")

## 6. Extension: year-to-year variability (SYNTHETIC)
Placeholder for a real 10-year NASA POWER / UNMA record. Gamma noise is added around each month's illustrative mean, scaled by the region's CV (`src/synthetic.py`). Swap `generate_all()` for a real data pull to get the real analysis.

In [ ]:
synthetic = generate_all(REGIONS, n_years=10, seed=42)
fig = plots.variability_boxplots(synthetic)
fig.savefig(FIG_DIR / "rainfall_boxplots_10yr_synthetic.png", dpi=150)
plt.show()